# Reinforcement Learning for Inventory / Stock Management
### Teaching an AI agent to make smart restocking decisions

---

## The Real-World Problem

Every shop, warehouse, or business faces the **inventory dilemma** daily:

> *"How much stock should I order today?"*

Order **too little** -> items run out -> lost sales, unhappy customers  
Order **too much**  -> excess stock -> storage costs, wastage  
Order **just right** -> maximum profit

This is exactly where **Reinforcement Learning** shines.

---

## Our Scenario: A Grain Store

A grain supplier manages stock for local farmers.
Every day, the manager (our RL agent) decides how much to reorder.

```
DAILY DECISION CYCLE
  Morning  : Check stock level        -> STATE
  Decide   : How many units to order? -> ACTION
  Day ends : Customers buy randomly
  Evening  : Count profit/loss        -> REWARD
  Next day : New stock level          -> NEXT STATE
```

---

## Problem Formulation

| RL Concept | Inventory Meaning |
|---|---|
| State   | Current stock level (0 to 50 units) |
| Action  | Order quantity (0, 5, 10, 15, or 20 units) |
| Reward  | Daily profit = Revenue - Order cost - Holding cost |
| Episode | One full month (30 days) |
| Goal    | Maximize total monthly profit |

### Reward Function:

```
Reward = (units_sold * sell_price)
       - (units_ordered * order_cost)
       - (stock_held * holding_cost)
       - stockout_penalty (if stock = 0)
```

| Parameter       | Value | Meaning              |
|----------------|-------|----------------------|
| Selling price  | 15    | Revenue per unit sold |
| Order cost     | 8     | Cost per unit ordered |
| Holding cost   | 1     | Storage per unit/day  |
| Stockout penalty | 50  | Lost trust per day    |

### Q-Learning Update Rule:

```
Q(s,a) <- Q(s,a) + alpha * [r + gamma * max Q(s',a') - Q(s,a)]
```

---

## Step 1: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from IPython.display import HTML, display
import random

np.random.seed(42)
random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

print("Libraries ready!")

## Step 2: Build the Inventory Environment

In [ ]:
class InventoryEnv:
    """
    Inventory Management Environment.

    Scenario: A grain store operates for 30 days (one episode).
    Each day the agent checks stock level and decides how much to reorder.
    Customer demand is random, simulating real-world uncertainty.

    Business Parameters:
        max_stock    : Maximum warehouse capacity (units)
        sell_price   : Revenue per unit sold
        order_cost   : Cost per unit ordered
        holding_cost : Storage cost per unit per day
        stockout_pen : Daily penalty when stock hits zero
        episode_len  : Number of days per episode (1 month)

    Reward = (units_sold * sell_price)
           - (order_qty  * order_cost)
           - (stock_held * holding_cost)
           - stockout_penalty (if stock = 0)
    """

    def __init__(
        self,
        max_stock    = 50,
        sell_price   = 15,
        order_cost   = 8,
        holding_cost = 1,
        stockout_pen = 50,
        episode_len  = 30,
        demand_mean  = 10,
        demand_std   = 4
    ):
        self.max_stock    = max_stock
        self.sell_price   = sell_price
        self.order_cost   = order_cost
        self.holding_cost = holding_cost
        self.stockout_pen = stockout_pen
        self.episode_len  = episode_len
        self.demand_mean  = demand_mean
        self.demand_std   = demand_std

        # States: discrete stock levels 0 to max_stock
        self.n_states = max_stock + 1

        # Actions: discrete order quantities
        self.order_options = [0, 5, 10, 15, 20]
        self.n_actions     = len(self.order_options)
        self.action_labels = [f'Order {x}' for x in self.order_options]

        self.stock   = 0
        self.day     = 0
        self.history = []

    def reset(self):
        """Start a new episode with random initial stock."""
        self.stock   = np.random.randint(5, 20)
        self.day     = 0
        self.history = []
        return self.stock  # State = current stock level

    def _sample_demand(self):
        """Sample random daily customer demand."""
        d = int(np.random.normal(self.demand_mean, self.demand_std))
        return max(0, d)

    def step(self, action_idx):
        """
        Execute one day:
          1. Receive ordered stock (clipped to warehouse capacity)
          2. Sample customer demand
          3. Sell as much as available
          4. Calculate reward
          5. Return (next_state, reward, done, info)
        """
        order_qty = self.order_options[action_idx]

        # Receive order
        self.stock = min(self.stock + order_qty, self.max_stock)

        # Customer demand
        demand = self._sample_demand()

        # Sell
        units_sold = min(demand, self.stock)
        lost_sales = demand - units_sold
        self.stock -= units_sold

        # Reward
        revenue      = units_sold * self.sell_price
        order_spend  = order_qty  * self.order_cost
        holding_fee  = self.stock * self.holding_cost
        stockout_fee = self.stockout_pen if (self.stock == 0 and demand > 0) else 0

        reward = revenue - order_spend - holding_fee - stockout_fee

        # Log
        self.history.append({
            'day':          self.day + 1,
            'stock_start':  self.stock + units_sold,
            'order_qty':    order_qty,
            'demand':       demand,
            'units_sold':   units_sold,
            'lost_sales':   lost_sales,
            'stock_end':    self.stock,
            'revenue':      revenue,
            'order_cost':   order_spend,
            'holding_cost': holding_fee,
            'stockout_fee': stockout_fee,
            'reward':       reward,
        })

        self.day += 1
        done = (self.day >= self.episode_len)

        return self.stock, reward, done, {
            'demand': demand, 'sold': units_sold,
            'lost_sales': lost_sales, 'order': order_qty
        }


# --- Test the environment ---
env   = InventoryEnv()
state = env.reset()

print("Grain Store Inventory Environment")
print("=" * 55)
print(f"  Max Capacity     : {env.max_stock} units")
print(f"  Sell Price       : {env.sell_price} per unit")
print(f"  Order Cost       : {env.order_cost} per unit")
print(f"  Holding Cost     : {env.holding_cost} per unit per day")
print(f"  Stockout Penalty : {env.stockout_pen} per day")
print(f"  Episode Length   : {env.episode_len} days")
print(f"  Actions          : {env.action_labels}")
print(f"  States           : {env.n_states} (stock levels 0-{env.max_stock})")
print()
print(f"Initial stock: {state} units")
print()
print(f"{'Day':>4} {'Order':>6} {'Demand':>7} {'Sold':>6} {'Stock':>6} {'Reward':>8}")
print("-" * 45)

for day in range(5):
    action = 2  # Always order 10 for demo
    next_state, reward, done, info = env.step(action)
    print(f"{day+1:>4} {info['order']:>6} {info['demand']:>7} "
          f"{info['sold']:>6} {next_state:>6} {reward:>8.1f}")
    state = next_state

## Step 3: Q-Learning Agent

In [ ]:
class QLearningAgent:
    """
    Q-Learning Agent for Inventory Management.

    Q-Table shape: (n_states, n_actions)
      - Rows = stock levels (0 to max_stock)
      - Cols = order quantities (0, 5, 10, 15, 20)

    The agent learns: given X units in stock, which order size maximizes profit?

    Update rule (Bellman equation):
      Q(s,a) <- Q(s,a) + alpha * [r + gamma * max Q(s',a') - Q(s,a)]
    """

    def __init__(
        self,
        n_states,
        n_actions,
        alpha         = 0.1,
        gamma         = 0.95,
        epsilon       = 1.0,
        epsilon_min   = 0.05,
        epsilon_decay = 0.997
    ):
        self.n_states      = n_states
        self.n_actions     = n_actions
        self.alpha         = alpha
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay

        self.q_table = np.zeros((n_states, n_actions))

        # Logs
        self.ep_rewards  = []
        self.ep_profits  = []
        self.ep_stockout = []
        self.epsilon_log = []

    def choose_action(self, state):
        """Epsilon-greedy: explore randomly OR exploit best known action."""
        state = min(state, self.n_states - 1)
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q_table[state]))

    def learn(self, state, action, reward, next_state, done):
        """Apply Bellman update to Q-table."""
        state      = min(state, self.n_states - 1)
        next_state = min(next_state, self.n_states - 1)

        current_q = self.q_table[state, action]
        if done:
            target_q = reward
        else:
            target_q = reward + self.gamma * np.max(self.q_table[next_state])

        self.q_table[state, action] += self.alpha * (target_q - current_q)

    def decay_epsilon(self):
        """Reduce exploration rate after each episode."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


env   = InventoryEnv()
agent = QLearningAgent(
    n_states      = env.n_states,
    n_actions     = env.n_actions,
    alpha         = 0.1,
    gamma         = 0.95,
    epsilon       = 1.0,
    epsilon_min   = 0.05,
    epsilon_decay = 0.997
)

print("Q-Learning Agent ready!")
print(f"\nHyperparameters:")
print(f"  alpha (learning rate)   = {agent.alpha}   -- how fast Q-values update")
print(f"  gamma (discount factor) = {agent.gamma}  -- long-term vs short-term profit")
print(f"  epsilon start           = {agent.epsilon}   -- start fully exploring")
print(f"  epsilon min             = {agent.epsilon_min}   -- always keep 5% exploration")
print(f"  epsilon decay           = {agent.epsilon_decay} -- slow decay for complex problem")
print(f"\nQ-Table shape: {agent.q_table.shape}")
print(f"  ({env.n_states} stock levels x {env.n_actions} order options)")

## Step 4: Train the Agent (1000 Episodes = 1000 Months)

In [ ]:
def train(env, agent, n_episodes=1000):
    print("Training started -- agent will simulate 1000 months of store operation\n")
    print(f"{'Episode':>8} {'Total Reward':>13} {'Avg Daily Profit':>17} "
          f"{'Stockout Days':>14} {'Epsilon':>9}")
    print("-" * 66)

    for ep in range(1, n_episodes + 1):
        state         = env.reset()
        total_reward  = 0
        stockout_days = 0
        done          = False

        while not done:
            action                         = agent.choose_action(state)
            next_state, reward, done, info = env.step(action)
            agent.learn(state, action, reward, next_state, done)
            state         = next_state
            total_reward += reward
            if info['lost_sales'] > 0:
                stockout_days += 1

        agent.decay_epsilon()
        agent.ep_rewards.append(total_reward)
        agent.ep_profits.append(total_reward / env.episode_len)
        agent.ep_stockout.append(stockout_days)
        agent.epsilon_log.append(agent.epsilon)

        if ep % 100 == 0 or ep == 1:
            print(f"{ep:>8} {total_reward:>13.1f} "
                  f"{total_reward/env.episode_len:>17.2f} "
                  f"{stockout_days:>14} {agent.epsilon:>9.4f}")

    print("-" * 66)
    last100_r = agent.ep_rewards[-100:]
    last100_s = agent.ep_stockout[-100:]
    print(f"\nTraining complete!")
    print(f"\nPerformance (last 100 episodes):")
    print(f"  Avg Monthly Profit  : {np.mean(last100_r):.1f}")
    print(f"  Avg Daily Profit    : {np.mean(last100_r)/env.episode_len:.2f}")
    print(f"  Avg Stockout Days   : {np.mean(last100_s):.1f} / 30 days")
    print(f"  Best Month          : {np.max(last100_r):.1f}")


train(env, agent, n_episodes=1000)

## Step 5: Training Visualizations

In [ ]:
def plot_training(agent, env, window=30):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Q-Learning Inventory Agent -- Training Metrics',
                 fontsize=16, fontweight='bold')

    eps = range(1, len(agent.ep_rewards) + 1)

    def moving_avg(data, w):
        return np.convolve(data, np.ones(w)/w, mode='valid')

    # --- 1. Monthly Profit ---
    ax = axes[0, 0]
    ax.plot(eps, agent.ep_rewards, alpha=0.2, color='#667EEA', lw=0.8)
    ma = moving_avg(agent.ep_rewards, window)
    ax.plot(range(window, len(eps)+1), ma, color='#553C9A', lw=2.5,
            label=f'{window}-ep moving avg')
    ax.set_title('Monthly Profit per Episode', fontweight='bold')
    ax.set_xlabel('Episode (Month)')
    ax.set_ylabel('Total Reward')
    ax.axhline(0, color='red', ls='--', alpha=0.4, label='Break-even')
    ax.legend()

    # --- 2. Stockout Days ---
    ax = axes[0, 1]
    ax.plot(eps, agent.ep_stockout, alpha=0.2, color='#FC8181', lw=0.8)
    ma_s = moving_avg(agent.ep_stockout, window)
    ax.plot(range(window, len(eps)+1), ma_s, color='#C53030', lw=2.5,
            label=f'{window}-ep moving avg')
    ax.set_title('Stockout Days per Month', fontweight='bold')
    ax.set_xlabel('Episode (Month)')
    ax.set_ylabel('Days with Unmet Demand')
    ax.axhline(2, color='orange', ls='--', alpha=0.6, label='Target: <=2 days')
    ax.legend()

    # --- 3. Epsilon Decay ---
    ax = axes[1, 0]
    ax.plot(eps, agent.epsilon_log, color='#F6AD55', lw=2.5)
    ax.fill_between(eps, agent.epsilon_log, alpha=0.15, color='#F6AD55')
    ax.set_title('Exploration (epsilon) Decay', fontweight='bold')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Epsilon')
    ax.set_ylim(0, 1.05)
    ax.axvspan(0, 300,   alpha=0.05, color='red',   label='Mostly exploring')
    ax.axvspan(600, 1000, alpha=0.05, color='green', label='Mostly exploiting')
    ax.legend(fontsize=8)

    # --- 4. Profit Distribution (first vs last 100 episodes) ---
    ax = axes[1, 1]
    first100 = agent.ep_rewards[:100]
    last100  = agent.ep_rewards[-100:]
    ax.hist(first100, bins=20, alpha=0.6, color='#FC8181', label='First 100 episodes')
    ax.hist(last100,  bins=20, alpha=0.6, color='#48BB78', label='Last 100 episodes')
    ax.axvline(np.mean(first100), color='#C53030', ls='--', lw=2,
               label=f'Early avg: {np.mean(first100):.0f}')
    ax.axvline(np.mean(last100),  color='#276749', ls='--', lw=2,
               label=f'Late avg: {np.mean(last100):.0f}')
    ax.set_title('Profit Distribution: Before vs After Training', fontweight='bold')
    ax.set_xlabel('Monthly Profit')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('training_metrics.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Training plots saved!")


plot_training(agent, env)

## Step 6: Learned Policy -- What Should the Store Order?

In [ ]:
def visualize_policy(env, agent):
    stock_levels  = list(range(0, env.max_stock + 1, 2))
    best_actions  = [np.argmax(agent.q_table[s]) for s in stock_levels]
    order_amounts = [env.order_options[a] for a in best_actions]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Learned Inventory Policy', fontsize=14, fontweight='bold')

    # --- Left: Policy Bar Chart ---
    ax = axes[0]
    colors_map = {0: '#FC8181', 5: '#F6AD55', 10: '#48BB78', 15: '#667EEA', 20: '#9F7AEA'}
    bar_colors = [colors_map[o] for o in order_amounts]
    ax.bar(stock_levels, order_amounts, color=bar_colors,
           edgecolor='white', linewidth=0.5, width=1.8)
    ax.set_title('Optimal Order Quantity by Stock Level', fontweight='bold')
    ax.set_xlabel('Current Stock Level (units)')
    ax.set_ylabel('Recommended Order Quantity (units)')
    ax.set_yticks(env.order_options)
    ax.axvspan(-1, 10, alpha=0.05, color='red')
    ax.axvspan(10, 30, alpha=0.05, color='yellow')
    ax.axvspan(30, 52, alpha=0.05, color='green')
    ax.text(4,  19, 'LOW\nSTOCK',    ha='center', fontsize=8, color='red',    fontweight='bold')
    ax.text(20, 19, 'MEDIUM\nSTOCK', ha='center', fontsize=8, color='#856404', fontweight='bold')
    ax.text(42, 19, 'HIGH\nSTOCK',   ha='center', fontsize=8, color='green',  fontweight='bold')
    legend_patches = [mpatches.Patch(color=colors_map[o], label=f'Order {o} units')
                      for o in env.order_options]
    ax.legend(handles=legend_patches, loc='upper right', fontsize=9)
    ax.set_xlim(-1, env.max_stock + 1)
    ax.set_ylim(0, 22)

    # --- Right: Q-Table Heatmap ---
    ax2 = axes[1]
    q_subset = agent.q_table[::2, :]
    im = ax2.imshow(q_subset, cmap='RdYlGn', aspect='auto')
    ax2.set_title('Q-Table Heatmap\n(Green = Agent prefers this order at this stock level)',
                  fontweight='bold')
    ax2.set_xlabel('Action (Order Quantity)')
    ax2.set_ylabel('Stock Level')
    ax2.set_xticks(range(env.n_actions))
    ax2.set_xticklabels(env.action_labels, rotation=15)
    ax2.set_yticks(range(len(stock_levels)))
    ax2.set_yticklabels([f'{s} units' for s in stock_levels], fontsize=7)
    plt.colorbar(im, ax=ax2, label='Q-Value (Expected Profit)')

    plt.tight_layout()
    plt.savefig('learned_policy.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Print readable policy table
    print("\nLEARNED POLICY SUMMARY")
    print("=" * 55)
    print(f"{'Stock Level':>12} | {'Rec. Order':>10} | Logic")
    print("-" * 55)
    for stock, order in zip(stock_levels, order_amounts):
        if stock <= 5:
            logic = 'Critically low -- order urgently'
        elif stock <= 15:
            logic = 'Low -- restock soon'
        elif stock <= 30:
            logic = 'Healthy -- moderate order'
        else:
            logic = 'Well-stocked -- minimal order'
        print(f"{stock:>12} | {order:>8} u | {logic}")


visualize_policy(env, agent)

## Step 7: Animate One Month of Smart Operations

In [ ]:
def run_episode_greedy(env, agent):
    """Run one full month with the trained agent (pure exploitation)."""
    state = env.reset()
    done  = False
    while not done:
        action = int(np.argmax(agent.q_table[min(state, env.n_states - 1)]))
        state, _, done, _ = env.step(action)
    return env.history


def animate_inventory(env, agent):
    history    = run_episode_greedy(env, agent)
    days       = [h['day']       for h in history]
    stocks     = [h['stock_end'] for h in history]
    orders     = [h['order_qty'] for h in history]
    demands    = [h['demand']    for h in history]
    rewards    = [h['reward']    for h in history]
    cum_profit = np.cumsum(rewards)

    total_profit   = sum(rewards)
    total_stockout = sum(1 for h in history if h['lost_sales'] > 0)
    print(f"Simulated Month Results (Trained Agent):")
    print(f"  Total Profit   : {total_profit:.1f}")
    print(f"  Daily Avg      : {total_profit/30:.2f}")
    print(f"  Stockout Days  : {total_stockout} / 30")
    print()

    fig = plt.figure(figsize=(15, 9))
    fig.patch.set_facecolor('#1A202C')
    gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    ax_stock  = fig.add_subplot(gs[0, :])
    ax_order  = fig.add_subplot(gs[1, 0])
    ax_reward = fig.add_subplot(gs[1, 1])
    ax_info   = fig.add_subplot(gs[1, 2])

    for ax in [ax_stock, ax_order, ax_reward, ax_info]:
        ax.set_facecolor('#2D3748')
        for spine in ax.spines.values():
            spine.set_edgecolor('#4A5568')

    fig.suptitle('Inventory Agent -- One Month of Smart Operations',
                 color='white', fontsize=14, fontweight='bold')

    def draw_frame(i):
        i = min(i, len(days) - 1)

        # Stock level chart
        ax_stock.clear()
        ax_stock.set_facecolor('#2D3748')
        ax_stock.set_xlim(0, 31)
        ax_stock.set_ylim(0, env.max_stock + 5)
        ax_stock.plot(days[:i+1], stocks[:i+1],
                      color='#90CDF4', lw=2.5, marker='o', ms=4, label='Stock Level')
        ax_stock.bar(days[:i+1], orders[:i+1],
                     alpha=0.35, color='#48BB78', label='Order Placed', width=0.6)
        ax_stock.step(days[:i+1], demands[:i+1],
                      color='#FC8181', lw=1.5, where='mid', label='Demand')
        ax_stock.axhline(10, color='orange', ls='--', lw=1, alpha=0.7,
                         label='Reorder point (10u)')
        ax_stock.set_title(
            f'Day {days[i]:2d}/30  --  Stock: {stocks[i]} units  |  '
            f'Demand: {demands[i]}  |  Order: {orders[i]}',
            color='white', fontsize=11)
        ax_stock.set_xlabel('Day', color='#A0AEC0')
        ax_stock.set_ylabel('Units', color='#A0AEC0')
        ax_stock.tick_params(colors='#A0AEC0')
        ax_stock.legend(loc='upper right', fontsize=8,
                        facecolor='#4A5568', labelcolor='white')
        if stocks[i] == 0:
            ax_stock.text(days[i], 2, 'STOCKOUT!', color='#FC8181',
                          fontsize=10, ha='center', fontweight='bold')

        # Order distribution
        ax_order.clear()
        ax_order.set_facecolor('#2D3748')
        order_counts = {o: orders[:i+1].count(o) for o in env.order_options}
        ax_order.bar(
            [str(o) for o in env.order_options],
            [order_counts[o] for o in env.order_options],
            color=['#FC8181', '#F6AD55', '#48BB78', '#667EEA', '#9F7AEA']
        )
        ax_order.set_title('Order Choices So Far', color='white', fontsize=10)
        ax_order.set_xlabel('Order Qty', color='#A0AEC0')
        ax_order.set_ylabel('# Days', color='#A0AEC0')
        ax_order.tick_params(colors='#A0AEC0')

        # Cumulative profit
        ax_reward.clear()
        ax_reward.set_facecolor('#2D3748')
        profit_color = '#48BB78' if cum_profit[i] >= 0 else '#FC8181'
        ax_reward.plot(days[:i+1], cum_profit[:i+1], color=profit_color, lw=2.5)
        ax_reward.fill_between(days[:i+1], cum_profit[:i+1],
                               alpha=0.2, color=profit_color)
        ax_reward.axhline(0, color='white', ls='--', lw=0.8, alpha=0.5)
        ax_reward.set_title('Cumulative Profit', color='white', fontsize=10)
        ax_reward.set_xlabel('Day', color='#A0AEC0')
        ax_reward.set_ylabel('Profit', color='#A0AEC0')
        ax_reward.tick_params(colors='#A0AEC0')
        ax_reward.text(0.5, 0.05, f'{cum_profit[i]:.0f}',
                       transform=ax_reward.transAxes, ha='center',
                       fontsize=12, fontweight='bold', color=profit_color)

        # Info panel
        ax_info.clear()
        ax_info.set_facecolor('#2D3748')
        ax_info.set_xlim(0, 1)
        ax_info.set_ylim(0, 1)
        ax_info.set_xticks([])
        ax_info.set_yticks([])
        ax_info.set_title("Today's Report", color='white', fontsize=10)
        h = history[i]
        stockout_so_far = sum(1 for d in history[:i+1] if d['lost_sales'] > 0)
        lines = [
            (0.85, f"Day          : {h['day']}/30",           '#90CDF4'),
            (0.73, f"Stock (end)  : {h['stock_end']} units",  '#90CDF4'),
            (0.61, f"Demand       : {h['demand']} units",     '#FBD38D'),
            (0.49, f"Sold         : {h['units_sold']} units", '#9AE6B4'),
            (0.37, f"Revenue      : {h['revenue']}",          '#9AE6B4'),
            (0.25, f"Today Profit : {h['reward']:.0f}",
             '#9AE6B4' if h['reward'] >= 0 else '#FC8181'),
            (0.13, f"Stockouts    : {stockout_so_far} days",
             '#FC8181' if stockout_so_far > 2 else '#9AE6B4'),
        ]
        for yp, txt, col in lines:
            ax_info.text(0.08, yp, txt, color=col, fontsize=10,
                         fontfamily='monospace', va='center')

    anim = animation.FuncAnimation(
        fig, draw_frame, frames=len(days), interval=500, repeat=True
    )
    html_anim = HTML(anim.to_jshtml())
    display(html_anim)
    anim.save('inventory_animation.gif', writer='pillow', fps=2)
    print("Animation saved as inventory_animation.gif")


animate_inventory(env, agent)

## Step 8: RL Agent vs Naive Strategies -- Who Wins?

In [ ]:
def run_strategy(env, strategy_fn, n_episodes=200):
    profits   = []
    stockouts = []
    for _ in range(n_episodes):
        state    = env.reset()
        total_r  = 0
        so_days  = 0
        done     = False
        while not done:
            action = strategy_fn(state, env)
            state, reward, done, info = env.step(action)
            total_r += reward
            if info['lost_sales'] > 0:
                so_days += 1
        profits.append(total_r)
        stockouts.append(so_days)
    return profits, stockouts


def always_order_10(state, env):  return 2
def random_strategy(state, env):  return np.random.randint(env.n_actions)
def rule_based(state, env):
    if state <= 5:    return 4
    elif state <= 15: return 3
    elif state <= 25: return 2
    elif state <= 35: return 1
    else:             return 0
def rl_agent_fn(state, env):
    return int(np.argmax(agent.q_table[min(state, env.n_states - 1)]))


print("Running comparison (200 episodes each)...\n")
strategies = {
    'Always Order 10': always_order_10,
    'Random Order':    random_strategy,
    'Rule-Based':      rule_based,
    'RL Agent (Ours)': rl_agent_fn,
}

results = {}
for name, fn in strategies.items():
    env_cmp = InventoryEnv()
    p, s    = run_strategy(env_cmp, fn, n_episodes=200)
    results[name] = {'profits': p, 'stockouts': s}
    print(f"  {name:<20} | Avg Profit: {np.mean(p):>7.1f} "
          f"| Avg Stockout: {np.mean(s):.1f} days")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Strategy Comparison: RL Agent vs Baselines',
             fontsize=14, fontweight='bold')

names  = list(results.keys())
colors = ['#CBD5E0', '#FC8181', '#F6AD55', '#48BB78']

ax = axes[0]
profit_data = [results[n]['profits'] for n in names]
bp = ax.boxplot(profit_data, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_xticklabels(names, rotation=12, ha='right')
ax.set_title('Monthly Profit Distribution', fontweight='bold')
ax.set_ylabel('Monthly Profit')

ax2 = axes[1]
avg_stockouts = [np.mean(results[n]['stockouts']) for n in names]
bars = ax2.bar(names, avg_stockouts, color=colors, edgecolor='white')
ax2.set_xticklabels(names, rotation=12, ha='right')
ax2.set_title('Average Stockout Days / Month', fontweight='bold')
ax2.set_ylabel('Days with Unmet Demand')
ax2.axhline(2, color='orange', ls='--', lw=1.5, label='Target: 2 days')
ax2.legend()
for bar, val in zip(bars, avg_stockouts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('strategy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Comparison complete!")

## Step 9: Decision Trace -- Why Did the Agent Order That?

In [ ]:
def trace_decisions(env, agent, n_days=10):
    state = env.reset()
    done  = False

    print("=" * 68)
    print("AGENT DECISION TRACE -- First 10 Days")
    print("=" * 68)

    for day in range(n_days):
        q_vals      = agent.q_table[min(state, env.n_states - 1)]
        best_action = int(np.argmax(q_vals))
        order_qty   = env.order_options[best_action]

        print(f"\nDay {day+1:2d} | Current Stock: {state:2d} units")
        print(f"  Q-Values:")
        for a, (label, q) in enumerate(zip(env.action_labels, q_vals)):
            marker = ' <-- BEST' if a == best_action else ''
            bar    = '#' * int(max(0, q) / 5)
            print(f"    {label:10s}: {q:8.2f}  {bar}{marker}")
        print(f"  Decision: Order {order_qty} units")

        state, reward, done, info = env.step(best_action)
        print(f"  Outcome : Demand={info['demand']}, Sold={info['sold']}, "
              f"Lost={info['lost_sales']}, Reward={reward:.0f}")
        if done:
            break


trace_decisions(env, agent)

print("\n")
print("=" * 55)
print("PROJECT SUMMARY")
print("=" * 55)
print("You built a Q-Learning agent that runs a grain store!")
print()
print("RL Concept       -> Business Meaning")
print("-" * 45)
print("State            -> Current stock level")
print("Action           -> How many units to reorder")
print("Reward           -> Daily profit")
print("Episode          -> One month of store operation")
print("Q-Table          -> Stock level -> best order qty")
print("Epsilon decay    -> Experimenting -> confident")
print()
print("What the agent learned:")
print("  - Order heavily when stock is critically low")
print("  - Order less when already well-stocked")
print("  - Avoid stockouts (costly penalty + lost revenue)")
print("  - Outperform fixed-order and rule-based strategies")

## Summary

---

### What you built:
A real-world Inventory Management system where a Q-Learning agent learns to run a grain store profitably -- from scratch, with no hardcoded rules.

### The RL -> Business mapping:

| RL Concept    | Business Meaning                          |
|---------------|-------------------------------------------|
| State         | Current stock level                       |
| Action        | How many units to reorder                 |
| Reward        | Daily profit (revenue - costs - penalty)  |
| Episode       | One month of store operation              |
| Q-Table       | Stock level -> best order quantity        |
| Epsilon decay | Experimenting -> confident decisions      |

### What the agent learned:
- Order heavily when stock is critically low
- Order less when already well-stocked (avoid holding costs)
- Avoid stockouts (they cost 50 penalty + lost revenue)
- Outperform fixed-order, random, and rule-based strategies

### Extend this project:
- Add seasonal demand (monsoon/summer spikes)
- Multi-product inventory (wheat, rice, fertilizer)
- Replace Q-Table with a neural network -> Deep Q-Network (DQN)
- Add supplier lead times (order takes 2 days to arrive)
- Connect to real sales data from a database

---
*Built as part of EPICS -- Farmer's Hub | Smart Agriculture Platform*